# NB01 · Introduction: Informatics, Chemical Space and the Drug-Discovery Problem
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/prateekpaul/Cheminformatics/blob/main/notebooks/nb01_chemical_space_and_drug_discovery.ipynb)

**Estimated time:** ~45 min | **Difficulty:** ●○○

An exploration of chemical space dimensions, combinatorial isomer expansion, the strategic choice between Structure-Based (SBDD) and Ligand-Based (LBDD) Drug Design, lead optimization as multi-parameter optimization (MPO), and the thermodynamic relationship between binding affinity ($K_i$) and free energy ($\Delta G^\circ$).

## Learning Objectives

By the end of this notebook, you will be able to:
1. **Define** cheminformatics and delineate its position within computational chemistry and biomedical discovery.
2. **Estimate** the astronomical scale of drug-like chemical space ($\sim 10^{60}$) in comparison to physical universe benchmarks.
3. **Model** constitutional isomer explosion in acyclic alkanes using Cayley graph enumeration theory.
4. **Evaluate** Structure-Based vs. Ligand-Based Drug Design workflows using decision trees.
5. **Formulate** drug discovery as a multi-parameter optimization (MPO) problem balancing potency, selectivity, and ADMET.
6. **Calculate** thermodynamic binding free energy ($\Delta G^\circ = RT \ln K_i$) across physiologically relevant affinity regimes.

## Prerequisites and Links

- NB00 (Setup and Python/RDKit Refresher) completed.
- Lecture 01 (Course Introduction: Chemical Space, SBDD vs LBDD, Free Energy).

In [ ]:
import os, random, warnings, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rdkit import Chem, RDLogger

SEED = 42
FULL_RUN = False
random.seed(SEED)
np.random.seed(SEED)
RDLogger.DisableLog("rdApp.*")
warnings.filterwarnings("ignore", category=DeprecationWarning)

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
print("NB01 setup completed.")

## 1. Defining Cheminformatics

The term **chemoinformatics** (or **cheminformatics**) was formally defined by F.K. Brown in 1998:

> *"Chemoinformatics is the mixing of those information resources to transform data into information and information into knowledge for the intended purpose of making better decisions faster in the arena of drug lead identification and optimization."* — Frank K. Brown (1998)

Cheminformatics bridges organic chemistry, computer science, and data analysis. It focuses on the representation, search, analysis, and property prediction of chemical structures at high throughput.

## 2. Chemical Space and Combinatorial Explosion

How vast is the space of small molecules? Bohacek et al. (1996) estimated that the number of synthetically accessible, drug-like organic molecules (molecular weight $\le 500$ Da, containing C, H, O, N, S, and halogens) is on the order of:

$$\mathcal{N}_{\text{drug-like}} \approx 10^{60}$$

To put this astronomical number in perspective:
- Estimated atoms in the Earth: $\sim 10^{50}$
- Total molecules synthesized by humanity (PubChem): $\sim 1.1 \times 10^8$
- Bioactive molecules with measured affinities (ChEMBL): $\sim 2.4 \times 10^6$
- Estimated atoms in the observable universe: $\sim 10^{80}$

### The Alkane Isomer Expansion

Even for simple saturated hydrocarbons (alkanes, $\text{C}_n\text{H}_{2n+2}$), the number of constitutional isomers grows exponentially with carbon count $n$ (Cayley tree enumeration, OEIS A000602):

| Carbons ($n$) | Formula | Constitutional Isomers |
|---|---|---|
| 1 | $\text{CH}_4$ | 1 |
| 4 | $\text{C}_4\text{H}_{10}$ | 2 |
| 6 | $\text{C}_6\text{H}_{14}$ | 5 |
| 8 | $\text{C}_8\text{H}_{18}$ | 18 |
| 10 | $\text{C}_{10}\text{H}_{22}$ | 75 |
| 20 | $\text{C}_{20}\text{H}_{42}$ | 366,319 |
| 30 | $\text{C}_{30}\text{H}_{62}$ | $4.11 \times 10^9$ |
| 40 | $\text{C}_{40}\text{H}_{82}$ | $6.25 \times 10^{13}$ |

> [!NOTE] Note vs. slides (Errata E13)
> The numbers above represent **constitutional (structural/skeletal) isomers** of acyclic alkanes. Stereoisomers (enantiomers arising from chiral carbons, e.g. 3-methylhexane) are excluded from these Cayley counts.

In [ ]:
# Plotting the exponential growth of alkane isomers
carbon_counts = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 14, 16, 18, 20]
isomer_counts = [1, 1, 1, 2, 3, 5, 9, 18, 35, 75, 355, 1858, 10359, 60523, 366319]

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(carbon_counts, isomer_counts, "o-", color="#1f77b4", lw=2, markersize=6)
ax.set_title("Combinatorial Explosion: Acyclic Alkane Constitutional Isomers", fontsize=12)
ax.set_xlabel("Number of Carbon Atoms ($n$)", fontsize=11)
ax.set_ylabel("Constitutional Isomers (Log Scale)", fontsize=11)
ax.grid(True, which="both", ls="--", alpha=0.5)
plt.tight_layout()
plt.show()

# Inspecting real alkane physical properties across isomers
csv_path = "../data/processed/d02_alkanes_bp.csv"
if not os.path.exists(csv_path):
    csv_path = "data/processed/d02_alkanes_bp.csv"

df_alkanes = pd.read_csv(csv_path)
print(df_alkanes[df_alkanes["n_carbons"] == 5][["name", "smiles", "boiling_point_C"]])

## 3. Structure-Based vs. Ligand-Based Drug Design

When developing a small-molecule drug, the starting computational strategy is governed by the structural knowledge available:

```mermaid
graph TD
    Start["New Drug Discovery Project"] --> Q1{"Is high-resolution 3D target structure known?"}
    Q1 -- Yes --> SBDD["Structure-Based Drug Design (SBDD)"]
    Q1 -- No --> Q2{"Are known active small-molecule ligands available?"}
    
    SBDD --> SBDD_Tools["Cavity Detection<br>Molecular Docking<br>Fragment-Based Design (FBDD)<br>De Novo Design"]
    
    Q2 -- Yes --> LBDD["Ligand-Based Drug Design (LBDD)"]
    Q2 -- No --> Target["Target Discovery & Modeling<br>AlphaFold / Cryo-EM / Homology"]
    
    LBDD --> LBDD_Tools["Pharmacophore Modeling<br>2D/3D Similarity Searching<br>QSAR / Machine Learning"]
```

### Strategic Comparison

| Characteristic | Structure-Based Drug Design (SBDD) | Ligand-Based Drug Design (LBDD) |
|---|---|---|
| **Prerequisites** | 3D experimental structure (X-ray, Cryo-EM, AlphaFold) | Bioactivity dataset of active/inactive molecules |
| **Core Methodologies** | Molecular docking, molecular dynamics, cavity mapping | QSAR, pharmacophores, fingerprint similarity |
| **Scoring Basis** | Physical/empirical force fields ($\Delta G_{\text{bind}}$) | Statistical correlation, similarity metrics (Tanimoto) |
| **Primary Risk** | Target flexibility, solvent entropic effects | Data bias, activity cliffs, lack of novelty |

## 4. Drug Design as Multi-Parameter Optimization (MPO)

A common misconception is that drug discovery is solely about maximizing binding affinity. In reality, high affinity is necessary but far from sufficient.

A clinical drug candidate must concurrently satisfy multiple competing criteria:
1. **Potency:** High on-target affinity ($K_i < 10$ nM).
2. **Selectivity:** Low affinity for antitargets (e.g. hERG potassium channel causing cardiac arrhythmia, CYP450 enzymes causing drug-drug interactions).
3. **Solubility & Permeability:** Aqueous solubility for gastrointestinal absorption and membrane permeability to reach intracellular targets.
4. **Metabolic Stability:** Appropriate clearance rate to allow once- or twice-daily dosing without hepatotoxicity.

In the 1990s, high-throughput screening often produced extremely lipophilic, potent molecules that failed in Phase II/III clinical trials due to poor pharmacokinetics and toxicity. This prompted Christopher Lipinski's famous **Rule of 5** (1997).

In [ ]:
# Multi-parameter scorecard comparison
categories = ['Potency (pKi)', 'Solubility', 'Permeability', 'Metabolic Stability', 'Safety (Anti-hERG)']
candidate_A = [9.0, 3.0, 4.0, 2.5, 3.0]  # Highly potent, but insoluble and unstable
candidate_B = [7.5, 8.0, 7.5, 8.0, 8.5]  # Moderately potent, excellent drug-like balance

angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
angles += angles[:1]
vals_A = candidate_A + candidate_A[:1]
vals_B = candidate_B + candidate_B[:1]

fig, ax = plt.subplots(figsize=(5, 5), subplot_kw=dict(polar=True))
ax.plot(angles, vals_A, color="#d62728", lw=2, label="Potent but Poor ADMET")
ax.fill(angles, vals_A, color="#d62728", alpha=0.15)
ax.plot(angles, vals_B, color="#2ca02c", lw=2, label="Balanced Lead Candidate")
ax.fill(angles, vals_B, color="#2ca02c", alpha=0.15)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=9)
ax.set_ylim(0, 10)
plt.title("Multi-Parameter Optimization Profile", y=1.08, fontsize=12)
plt.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=9)
plt.tight_layout()
plt.show()

## 5. Thermodynamics of Binding: From $K_i$ to $\Delta G^\circ$

At thermodynamic equilibrium, the binding of a ligand $L$ to a protein receptor $R$:

$$R + L \rightleftharpoons RL$$

is governed by the dissociation constant $K_i$ (or $K_d$):

$$K_i = \frac{[R][L]}{[RL]}$$

The standard Gibbs free energy of binding $\Delta G^\circ_{\text{bind}}$ is related to $K_i$ via:

$$\Delta G^\circ = RT \ln K_i = -RT \ln K_a$$

where:
- $R = 8.3144626\ \text{J}/(\text{mol}\cdot\text{K}) = 1.9872 \times 10^{-3}\ \text{kcal}/(\text{mol}\cdot\text{K})$
- $T = 298.15\ \text{K}$ (standard laboratory temperature, $25^\circ\text{C}$)
- At $298.15$ K, $RT \approx 2.479\ \text{kJ/mol} \approx 0.5924\ \text{kcal/mol}$.

### The Logarithmic Rule of Thumb

Because $\ln(10) \approx 2.3026$:

$$\Delta G^\circ = 2.3026 \times RT \times \log_{10}(K_i)$$

At $298.15\ \text{K}$, **every 10-fold improvement in binding affinity** (1 unit on the $pK_i = -\log_{10} K_i$ scale) corresponds to:

$$\Delta \Delta G^\circ \approx -5.71\ \text{kJ/mol} \approx -1.36\ \text{kcal/mol}$$

This means a single well-placed hydrogen bond (worth $\sim 2\text{--}5$ kcal/mol) can improve affinity by **100- to 1,000-fold**!

In [ ]:
R_KJ = 8.3144626e-3  # kJ / (mol * K)
R_KCAL = 1.9872e-3    # kcal / (mol * K)
T_STD = 298.15        # K

affinities = [
    ("Millimolar (mM)", 1e-3),
    ("Micromolar (uM)", 1e-6),
    ("Nanomolar (nM)",  1e-9),
    ("Picomolar (pM)",  1e-12),
    ("Femtomolar (fM)", 1e-15)
]

free_energy_table = []
for name, ki in affinities:
    dG_kj = R_KJ * T_STD * math.log(ki)
    dG_kcal = R_KCAL * T_STD * math.log(ki)
    free_energy_table.append({
        "Affinity Regime": name,
        "Ki (Molar)": f"{ki:.0e}",
        "pKi (-log10 Ki)": -math.log10(ki),
        "Delta G (kJ/mol)": round(dG_kj, 2),
        "Delta G (kcal/mol)": round(dG_kcal, 2)
    })

pd.DataFrame(free_energy_table)

## 6. Deterministic vs. Data-Driven Approaches

Computational cheminformatics deploys two complementary paradigms:

| Metric | Deterministic (Physics-Based) | Data-Driven (Statistical / ML) |
|---|---|---|
| **Underlying Theory** | Quantum mechanics (DFT), Newton's laws (MD, force fields) | Statistical learning, inductive bias, deep neural networks |
| **Input Requirements** | Explicit 3D atomic coordinates, force field parameters | 2D molecular graphs, SMILES strings, bioassay activity tables |
| **Computational Cost** | High (CPU/GPU-intensive, minutes to days per ligand) | Low (milliseconds to seconds per molecule) |
| **Generalization** | Transferable across unseen chemotypes | High fidelity within training domain; vulnerable to activity cliffs |
| **Examples** | Free Energy Perturbation (FEP), AutoDock Vina, Desmond | Random Forest QSAR, ChemBERTa, Graph Neural Networks |

### Exercise 1.1

Write a function `compute_delta_g(ki_molar: float, temperature_k: float = 298.15, unit: str = "kJ/mol") -> float` that calculates standard binding free energy $\Delta G^\circ = RT \ln K_i$. Support units `'kJ/mol'` and `'kcal/mol'`. If an invalid unit is given or $K_i \le 0$, raise a `ValueError`.

In [ ]:
def compute_delta_g(ki_molar: float, temperature_k: float = 298.15, unit: str = "kJ/mol") -> float:
    # YOUR CODE HERE
    raise NotImplementedError("Implement compute_delta_g")

In [ ]:
# Self-check for Exercise 1.1
# Check 10 uM, 10 nM, 10 pM
dg_10um = compute_delta_g(1e-5, 298.15, "kJ/mol")
dg_10nm = compute_delta_g(1e-8, 298.15, "kJ/mol")
dg_10pm = compute_delta_g(1e-11, 298.15, "kJ/mol")

assert math.isclose(dg_10um, -28.537, abs_tol=0.05)
assert math.isclose(dg_10nm, -45.659, abs_tol=0.05)
assert math.isclose(dg_10pm, -62.781, abs_tol=0.05)

# Check kcal/mol
dg_10nm_kcal = compute_delta_g(1e-8, 298.15, "kcal/mol")
assert math.isclose(dg_10nm_kcal, -10.91, abs_tol=0.05)
print('✔ Exercise 1.1 passed!')

### Exercise 1.2

Write an inverse function `ki_from_delta_g(delta_g: float, temperature_k: float = 298.15, unit: str = "kJ/mol") -> float` that computes the equilibrium dissociation constant $K_i$ in Molar from a given $\Delta G^\circ$.

In [ ]:
def ki_from_delta_g(delta_g: float, temperature_k: float = 298.15, unit: str = "kJ/mol") -> float:
    # YOUR CODE HERE
    raise NotImplementedError("Implement ki_from_delta_g")

In [ ]:
# Self-check for Exercise 1.2
ki_orig = 1e-8
dg = compute_delta_g(ki_orig, 298.15, "kJ/mol")
ki_recov = ki_from_delta_g(dg, 298.15, "kJ/mol")
assert math.isclose(ki_orig, ki_recov, rel_tol=1e-5)
print('✔ Exercise 1.2 passed!')

### Exercise 1.3

Write a function `get_alkane_isomer_count(n_carbons: int) -> int` that returns the exact number of acyclic alkane constitutional isomers for $n \in [1, 8]$ according to Cayley's theorem (OEIS A000602). For $n$ outside $[1, 8]$, raise a `ValueError`.

In [ ]:
def get_alkane_isomer_count(n_carbons: int) -> int:
    # YOUR CODE HERE
    raise NotImplementedError("Implement get_alkane_isomer_count")

In [ ]:
# Self-check for Exercise 1.3
assert get_alkane_isomer_count(1) == 1
assert get_alkane_isomer_count(4) == 2
assert get_alkane_isomer_count(6) == 5
assert get_alkane_isomer_count(7) == 9
assert get_alkane_isomer_count(8) == 18
print('✔ Exercise 1.3 passed!')

## Common Pitfalls

1. **Natural vs. Base-10 Logarithms:** $\Delta G^\circ = RT \ln K_i$, using the natural logarithm (`math.log`). Using $\log_{10}$ produces errors by a factor of $2.303$.
2. **Affinity Units in Free Energy Equations:** $K_i$ must always be converted to standard-state **Molar units (M)** before taking the logarithm. Passing nanomolar concentrations ($10$ instead of $10^{-8}$) yields an erroneous positive free energy!
3. **Equating Constitutional Isomers with Stereoisomers:** Acyclic alkane isomer tables enumerate constitutional (connectivity) isomers. Real compounds with stereocenters have additional stereoisomeric configurations.

## Summary and Key Takeaways

- Cheminformatics focuses on data-driven chemical information extraction and decision-making in lead discovery.
- Drug-like chemical space is virtually inexhaustible ($\sim 10^{60}$), demanding algorithmic filtering and generative search.
- SBDD is chosen when a 3D target structure exists; LBDD is deployed when active small-molecule data are available.
- Drug discovery is an inherently multi-objective optimization problem; potency without acceptable ADMET leads to clinical failure.
- Binding free energy is logarithmic: a 10-fold increase in binding affinity represents only $-5.71$ kJ/mol ($-1.36$ kcal/mol) of binding free energy at 298 K.

## References

- Bohacek, R. S. et al. The art and practice of structure-based drug design: A molecular modeling perspective. *Med. Res. Rev.* **1996**, 16, 3–50.
- Brown, F. K. Chemoinformatics: What is it and how does it impact drug discovery? *Annu. Rep. Med. Chem.* **1998**, 33, 375–384.
- Lipinski, C. A. et al. Experimental and computational approaches to estimate solubility and permeability in drug discovery and development settings. *Adv. Drug Deliv. Rev.* **1997**, 23, 3–25.

In [ ]:
# Self-test check
print("✔ NB01 checks passed")